In [4]:
from pathlib import Path
import os

# Notebook está en RAG/notebooks - subimos un nivel
ROOT = Path.cwd().parent
os.chdir(ROOT)

print("Project root:", Path.cwd())

import pandas as pd
from src.pipeline import answer_question


Project root: c:\Users\nicol\OneDrive\Documentos\Cursos\RAG


c:\Users\nicol\OneDrive\Documentos\Cursos\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1837.52it/s]


In [ ]:
qa = pd.read_json("data/raw/financebench/financebench_open_source.jsonl", lines=True)

SAMPLE_PER_TYPE = 10
sample = qa.groupby("question_type", group_keys=False).sample(n=SAMPLE_PER_TYPE, random_state=42)

results = []
for _, row in sample.iterrows():
    resultado = answer_question(row["question"])
    expected_pages = [e["evidence_page_num"] for e in row["evidence"]]
    retrieved_pages = [s["page_num"] for s in resultado["sources"]]

    results.append({
        "financebench_id": row["financebench_id"],
        "question_type": row["question_type"],
        "company": row["company"],
        "question": row["question"],
        "expected_answer": row["answer"],
        "generated_answer": resultado["answer"],
        "expected_evidence_pages": expected_pages,
        "retrieved_pages": retrieved_pages,
        "evidence_page_found": any(p in retrieved_pages for p in expected_pages),
    })
    print(f"[{row['financebench_id']}] listo")


[empresa='AMD', año=2022, filtro_aplicado=True]
[financebench_id_01198] listo
[empresa='Microsoft', año=2023, filtro_aplicado=True]
[financebench_id_00552] listo
[empresa='Corning', año=2022, filtro_aplicado=True]
[financebench_id_00005] listo
[empresa='Ulta Beauty', año=2023, filtro_aplicado=True]
[financebench_id_00746] listo
[empresa='General Mills', año=2020, filtro_aplicado=True]
[financebench_id_03471] listo
[empresa='Microsoft', año=2016, filtro_aplicado=True]
[financebench_id_04700] listo
[empresa='Activision Blizzard', año=2019, filtro_aplicado=True]
[financebench_id_02987] listo
[empresa='PepsiCo', año=2022, filtro_aplicado=True]
[financebench_id_04481] listo
[empresa='3M', año=None, filtro_aplicado=True]
[financebench_id_01858] listo
[empresa='Verizon', año=2021, filtro_aplicado=True]
[financebench_id_02024] listo
[empresa='Amcor', año=2023, filtro_aplicado=True]
[financebench_id_01928] listo
[empresa='Pfizer', año=2023, filtro_aplicado=True]
[financebench_id_02419] listo


In [6]:
df_results = pd.DataFrame(results)
df_results.to_json("data/processed/pilot_results.jsonl", orient="records", lines=True)

print("\n=== Resumen ===")
print(f"Preguntas evaluadas: {len(df_results)}")
print(f"Página de evidencia recuperada: {df_results['evidence_page_found'].sum()}/{len(df_results)}")
print("Revisa manualmente generated_answer vs expected_answer en data/processed/pilot_results.jsonl")


=== Resumen ===
Preguntas evaluadas: 12
Página de evidencia recuperada: 7/12
Revisa manualmente generated_answer vs expected_answer en data/processed/pilot_results.jsonl
